# modifier

In [ ]:
# pretty_modifier.py
# A beauty-focused OVITO Python modifier:
# - Computes a metric per particle (distance from origin, but you can swap it)
# - Writes 'Radius' (nice, robust scaling)
# - Writes 'Color' (N x 3 float array, 0..1)
# - Writes 'Transparency' (0..1)
# - Stores summary stats in data.attributes for quick inspection

import numpy as np

def _smooth_gradient01(t):
    """
    t in [0,1] -> RGB in [0,1]
    A perceptually pleasant 4-stop gradient: blue -> cyan -> yellow -> red.
    """
    # Segment positions (0..1) and colors (RGB)
    stops = np.array([0.00, 0.33, 0.66, 1.00], dtype=np.float64)
    cols  = np.array([
        [0.10, 0.25, 0.95],  # blue
        [0.10, 0.85, 0.95],  # cyan
        [0.98, 0.92, 0.20],  # yellow
        [0.92, 0.15, 0.10],  # red
    ], dtype=np.float64)

    t = np.clip(t, 0.0, 1.0)
    # Find segment indices
    idx = np.minimum(np.searchsorted(stops, t, side="right") - 1, len(stops) - 2)
    left = stops[idx]
    right = stops[idx + 1]
    w = (t - left) / (right - left + 1e-12)
    c = (1.0 - w)[:, None] * cols[idx] + w[:, None] * cols[idx + 1]
    return c

def pretty_modifier(frame, data):
    if data.particles is None:
        return

    # === Source positions ===
    pos_prop = data.particles.positions
    coords = pos_prop.array  # (N, 3)

    # === Beauty metric ===
    # Use distance from origin; swap this for any scalar you like (e.g., coordination #)
    r = np.linalg.norm(coords, axis=1)  # shape (N,)

    # === Robust normalization (percentile based) ===
    p1, p99 = np.percentile(r, [1.0, 99.0])
    span = max(p99 - p1, 1e-12)
    r01 = np.clip((r - p1) / span, 0.0, 1.0)  # robust [0,1]

    # === Particle Radius (visual size) ===
    # Map r01 -> radius with gentle easing for nicer distribution
    # Final radius ~ [min_rad, max_rad]
    min_rad, max_rad = 0.25, 1.35
    eased = r01 ** 0.6
    radii = np.full_like(eased, 1.8) # min_rad + (max_rad - min_rad) * eased
    data.particles_.create_property('Radius', data=radii.astype(np.float64))

    # === Particle Color (N x 3) ===
    colors = _smooth_gradient01(r01).astype(np.float64)
    data.particles_.create_property('Color', data=colors)

    # === Subtle Transparency for depth cueing (0=opaque, 1=transparent) ===
    # Near = more opaque, far = slightly more transparent
    trans = 0.10 + 0.35 * r01  # range ~ 0.10..0.45
    data.particles_.create_property('Transparency', data=np.clip(trans, 0.0, 0.75))

    # === Handy attributes for overlays / debugging ===
    data.attributes['BeautyMetric_Min'] = float(r.min())
    data.attributes['BeautyMetric_Max'] = float(r.max())
    data.attributes['BeautyMetric_Mean'] = float(r.mean())
    data.attributes['Beauty_Normalized_P1'] = float(p1)
    data.attributes['Beauty_Normalized_P99'] = float(p99)

In [ ]:
# demo_pipeline.py
# Run with OVITO's Python (recommended):  ovitos demo_pipeline.py
# Or with standard Python if you installed the 'ovito' package: python demo_pipeline.py

from ovito.io import import_file
from ovito.modifiers import PythonScriptModifier
from ovito.io import import_file
from ovito.modifiers import PythonScriptModifier, ColorCodingModifier
from ovito.vis import Viewport, TachyonRenderer, ColorLegendOverlay

# 1) Import a structure/trajectory file.
# Replace with your path and format (OVITO auto-detects many formats: LAMMPS dump, XYZ, POSCAR, etc.)
input_path = "../jobs/session-atomagent-demo1/output/dump.shake2.lammpstrj"
pipeline = import_file(input_path)

# 2) Add your custom Python modifier (defined in my_modifier.py).
pipeline.modifiers.append(PythonScriptModifier(function=pretty_modifier))


# (Optional) Evaluate once to force computation and check attributes/properties.
data = pipeline.compute()
print("MeanRadius:", data.attributes.get("MeanRadius"))

# 3) Render an image.
# Add the pipeline to the scene so the viewport can render it:
node = pipeline.add_to_scene()

import math

vp = Viewport()
vp.type = Viewport.Type.Perspective           # free camera
vp.camera_pos = (10, 10, 0)                 # move the camera
vp.camera_dir = (-1, -1, 0)                # look toward the origin (target - camera_pos)
vp.camera_up  = (0, 0, 1)                     # z-up scenes look nice for atomistic data
vp.fov = math.radians(42)                     # a bit tighter than default
vp.zoom_all()                # frame the scene with this aspect ratio# auto-frame the scene if needed


# --- Make the simulation-box lines thinner ---
cell_vis = pipeline.source.data.cell.vis
cell_vis.line_width = 0.25        # in simulation units (reduce if still too bold)
cell_vis.rendering_color = (0,0,0)  # keep black; or try a soft gray like (0.15,0.15,0.15)

renderer = TachyonRenderer()
out_path = "render.png"

from ovito.qt_compat import QtCore
# legend = ColorLegendOverlay(
#     title = 'Potential energy per atom',
#     pipeline=pipeline,
#     property = 'Position.Z',  # must match the property used in ColorCodingModifier
#     alignment = QtCore.Qt.AlignmentFlag.AlignLeft | QtCore.Qt.AlignmentFlag.AlignTop,
#     orientation = QtCore.Qt.Orientation.Vertical,
#     offset_y = -10,
#     offset_x= 10,
#     font_size = 5,
#     format_string = '%.2f')
# vp.overlays.append(legend)
num_frame = pipeline.source.num_frames
vp.render_image(
    size=(2000, 9000),
    filename=out_path,
    renderer=renderer,
    background=(1, 1, 1)           # white background
)
print(f"Rendered to {out_path}")

In [ ]:
num_frame = pipeline.source.num_frames
out_path = "render_final.png"
img = vp.render_image(
    size=(2000, 10000),
    filename=out_path,
    renderer=renderer,
    background=(1, 1, 1),          # white background
    frame=num_frame - 1         # last frame (0..num_frame-1
)

In [ ]:
vp.render_anim(
    size=(1920, 9000),  # Full HD
    filename="movie.mp4",
    renderer=renderer,
    fps=1
)